# Agentic AI Workshop — Day 1
## How LLMs work and your first Groq API call

This notebook follows the uploaded Day 1 workshop: first connect to a model, then build a fact-grounded Sweet Crumbs bakery assistant.

**Run from top to bottom.** Store your key in Colab Secrets; never paste it into a code cell.

## 1. Install the client library

Groq exposes an OpenAI-compatible API. We use the `openai` Python package and configure its `base_url` to point to Groq.

In [ ]:
!pip -q install --upgrade openai

## 2. Load your Groq API key securely

In Colab, click the key icon on the left → add a secret named `GROQ_API_KEY` → paste your key → enable Notebook access.

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "GROQ_API_KEY was not found. Check Colab Secrets."

## 3. Configure the Groq client

`OpenAI(...)` configures the client. The network request happens later when we call `.create(...)`.

In [ ]:
from openai import OpenAI

MODEL_NAME = "openai/gpt-oss-20b"  # If unavailable, use a current chat model ID from Groq's model catalogue.

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

print("Groq client configured.")

## 4. Your first API call

The `messages` list has a system instruction and a user message. The assistant's generated text is returned inside `choices[0].message.content`.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a funny joker."},
        {"role": "user", "content": "Tell me a short joke."},
    ],
)

print(response.choices[0].message.content)
print("\nToken usage:", response.usage)

## 5. Read the response object

The response contains more than the text. Inspect the model, finish reason and usage.

In [ ]:
print("Model:", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("Prompt tokens:", response.usage.prompt_tokens if response.usage else "not reported")
print("Completion tokens:", response.usage.completion_tokens if response.usage else "not reported")

## 6. Build: Sweet Crumbs Home Bakery assistant

The workshop's sample business facts are fictional:
- Orders need 24 hours' notice.
- Eggless versions are available.
- Plum cake ₹500, chocolate cake ₹650, cupcake ₹80.
- Do not promise delivery times.
- If a fact is missing, say you will check with the owner.

In [ ]:
SYSTEM_PROMPT = """ROLE:
You are the customer-support assistant for Sweet Crumbs Home Bakery.

FACTS:
- We bake to order.
- Orders need 24 hours' notice.
- All cakes have eggless versions.
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80.

RULES:
- Never promise a delivery time.
- Do not invent delivery charges, stock, discounts or policies.
- If the answer is not present in FACTS, say: "I'll check with the owner."

STYLE:
- Be friendly and concise.
- Reply in English or Telugu according to the customer's language.
- Keep the answer under 50 words.
"""

bakery_response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Can I get an eggless cake today evening?"},
    ],
)

print(bakery_response.choices[0].message.content)
print("\nUsage:", bakery_response.usage)

### Try your own bakery questions

Change the question and rerun the cell. Check whether the answer follows the facts and refuses to invent missing information.

In [ ]:
customer_question = "What is the price of a chocolate cake?"
test_response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": customer_question},
    ],
)
print(test_response.choices[0].message.content)

## 7. Temperature experiment

Low temperature is useful for factual, consistent tasks. Higher temperature can produce more varied wording. It does not make an answer more factual.

In [ ]:
tagline_prompt = "Write one short, memorable tagline for Sweet Crumbs Home Bakery."

for temperature in (0.2, 1.2):
    print(f"\\n--- temperature={temperature} ---")
    for i in range(3):
        result = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a creative bakery copywriter."},
                {"role": "user", "content": tagline_prompt},
            ],
        )
        print(f"{i+1}. {result.choices[0].message.content}")

## 8. Memory experiment

An API call does not automatically include the previous call's conversation. To give the model context, send the earlier messages again.

In [ ]:
# Call 1: the model is told a name.
first_turn = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "My name is Ravi."}],
)
print("Call 1:", first_turn.choices[0].message.content)

# Call 2: no earlier history is sent.
second_turn = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nCall 2 without history:", second_turn.choices[0].message.content)

# Call 3: send the conversation history explicitly.
third_turn = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "My name is Ravi."},
        {"role": "assistant", "content": first_turn.choices[0].message.content},
        {"role": "user", "content": "What is my name?"},
    ],
)
print("\nCall 3 with history:", third_turn.choices[0].message.content)

## 9. Your 10% — Day 1 practice

1. Compare three tagline outputs at temperatures `0.2` and `1.2`.
2. Remove the FACTS from the system prompt and ask about delivery to Kakinada. Observe whether the model guesses.
3. Compare English and Telugu prompt token usage using `response.usage`.
4. Write five verified facts and three rules for a business you know. Replace `SYSTEM_PROMPT`.
5. Test at least three real customer questions. Check every answer yourself.

**Before Day 2:** write down the ten most common questions customers ask your chosen business.

## Troubleshooting
- Missing key: check the exact Colab secret name `GROQ_API_KEY` and Notebook access.
- 401: check that the key is a Groq key and that the base URL is Groq's.
- 404: check the current Groq model catalogue and update `MODEL_NAME`.
- 429: wait and reduce repeated requests.

Never publish your API key or commit it to GitHub.